In [ ]:
%%capture
import os
import pandas as pd
from dj_notebook import activate
from pathlib import Path
env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
from intecomm_analytics.dataframes import get_appt_df, get_all_complications_df, get_medications_df
from edc_pdutils.dataframes import get_crf
from intecomm_analytics.dataframes.main_1858_to_stata import df_main_variable_labels
from edc_analytics.stata import get_stata_labels_from_model

In [ ]:
df_appt = get_appt_df()

In [ ]:
variable_labels = {}
system_columns = ["id", "consent_model", "consent_version", "crf_status", "crf_status_comments", "created", "modified", "user_created", "user_modified", "hostname_created", "hostname_modified", "device_created", "device_modified", "locale_created", "locale_modified", "revision"]
df_appt_columns = [col for col in df_appt.columns.tolist() if col not in ["subject_visit_id", "appointment_id"]]

In [ ]:
df = df_appt.copy().reset_index(drop=True)

In [ ]:
df_complications, df_complications_variable_labels = get_all_complications_df()
df = df.merge(df_complications, on="subject_visit_id", how="left", suffixes=("", "_y"))
df = df.drop(columns=[col for col in df.columns if col.endswith("_y")])
assert len(df) == 20953
variable_labels.update(**df_complications_variable_labels)

In [ ]:
df_meds, medication_variable_labels = get_medications_df()
df = df.merge(df_meds, on="subject_visit_id", how="left", suffixes=("", "_y"))
df = df.drop(columns=[col for col in df.columns if col.endswith("_y")])
assert len(df) == 20953
variable_labels.update(**medication_variable_labels)

In [ ]:
df_vitals = get_crf(model="intecomm_subject.vitals",subject_visit_model="intecomm_subject.subjectvisit", read_verbose=False)
df_vitals = df_vitals.sort_values(["subject_identifier", "visit_code"])
df_vitals[["weight", "height"]] = df_vitals.groupby("subject_identifier")[["weight", "height"]].ffill()
df_vitals["bmi"] = (df_vitals["weight"]) / ((df_vitals["height"] / 100) ** 2)
df = df.merge(df_vitals[["subject_visit_id", "weight","temperature", "height", "bmi", "sys_blood_pressure_avg", "dia_blood_pressure_avg", "severe_htn"]], on="subject_visit_id", how="left", suffixes=("_x", ""))
df = df.drop(columns=[col for col in df.columns if col.endswith("_x")])
variable_labels.update(**get_stata_labels_from_model(df_appt, f"intecomm_subject.vitals", None))

In [ ]:
assert len(df) == 20953

In [ ]:
assert (
    len(
        df_meds[
            (df_meds["subject_visit_id"].duplicated(keep=False))
            & (df_meds["subject_visit_id"].notna())
        ]
    )
    == 0
)


In [ ]:
variable_labels.update(**get_stata_labels_from_model(df, model="edc_appointment.appointment", suffix=None))
variable_labels.update(**df_main_variable_labels())
df.to_stata(
    path=analysis_folder / "complications.dta",
    variable_labels=variable_labels,
    version=118,
    write_index=False,
)